# Let's talk about Multi-Model foundation models

### 1. The Challenge
> 🚩 **Problem:** Using an *individual model for individual tasks* is inefficient and difficult to train and deploy.

### 2. Why Multi-Model Foundation Models?
*   🧠 **Better Generalization**
*   🚀 **Faster Inference**
*   🧩 **Adaptability:** Solve different tasks without retraining
*   🌐 **Scale:** Training with hundreds of millions of image-text pairs from the internet

### 3. Foundation Models Introduction

| Category | Models |
| :--- | :--- |
| **Language Model** | ELMo, BERT, GPT, T5 |
| **Classification** | **CLIP** *(fully researched & ready)*, CoCa |
| **LM + Vision** | LLaVA, Flamingo, GPT-4V, Gemini, Molmo |
| **Generative & More** | Segment Anything, Whisper, DALL·E, Stable Diffusion, Imagen |
| **Chaining** | LMs + CLIP, Visual Programming |

### 4. Data Flow of Significant Networks

#### 4.1 CLIP (Contrastive Language-Image Pre-training)

**Feature Encoding:**
*   🖼️ **Image:** $\text{Input} \to \text{Encoder (ResNet/ViT)} \to \text{Features} \to \text{Linear Proj} \to I_e$
*   📝 **Text:** $\text{Input} \to \text{Encoder (BERT)} \to \text{Features} \to \text{Linear Proj} \to T_e$

**Similarity & Loss:**
$$
\begin{aligned}
\text{logits}_{\text{image}} &= (I_e @ T_e^T) \cdot \exp(\tau) \\
\text{logits}_{\text{text}} &= (T_e @ I_e^T) \cdot \exp(\tau)
\end{aligned}
$$

**Probabilities:**
$$
\begin{aligned}
P_{\text{image}} &= \text{softmax}(\text{logits}_{\text{image}}, \text{dim}=1) \\
P_{\text{text}} &= \text{softmax}(\text{logits}_{\text{text}}, \text{dim}=1)
\end{aligned}
$$

#### 4.2 LLaVA (Large Language-and-Vision Assistant)

**1. High-Level Concept**
*   **Flow:** $\text{Image Tokens} + \text{Input Text} \to \text{Transformer Block} \to \text{Text Completion}$
*   **❓ Question:** What kind of input image tokens work best?
*   **💡 Answer:** Image tokens generated by the **CLIP** image encoder.

**2. Detailed Data Flow**

$$
\text{Input Image} \xrightarrow{\text{Patching}} \text{Patches} \xrightarrow[\text{+ Pos Embed}]{\text{Flatten + Proj}} \text{ViT Block} \xrightarrow{\text{Features}^*} \text{Linear Layer} \to \text{LLM Input Tokens}
$$

> *\*Note: In practice, features extracted before the final transformer layer are often more effective.*

#### 4.3 Flamingo (Visual Language Model)

**1. High-Level Concept**
*   **Flow:** $\text{Interleaved Images} + \text{Text} \to \text{Perceiver Resampler} \to \text{Frozen LLM} \xrightarrow{\text{Gated X-Attn}} \text{Response}$
*   **❓ Question:** How to fuse visual features into a fixed-size input for a frozen LLM?
*   **💡 Answer:** Use a **Perceiver Resampler** to fix feature size and **Gated Cross-Attention** to inject them.

**2. Detailed Data Flow**

$$
\text{Vision Features} \xrightarrow[\text{+ Latent Queries}]{\text{Perceiver Resampler}} \text{Fixed Visual Tokens} \xrightarrow{\text{Keys/Values}} \text{Gated X-Attn} \xrightarrow[\text{Text Query}]{\text{Frozen LLM Layer}} \text{Output}
$$

> *Note: Flamingo keeps the Vision Encoder and LLM frozen, training only the adapter layers (Resampler & Gated X-Attn).*

#### 4.4 Molmo (Open Weights & Data VLM)

**1. High-Level Concept**
*   **Flow:** $\text{Image} + \text{Text} \to \text{Vision Encoder} \to \text{Pooling/Projection} \to \text{LLM} \to \text{Response}$
*   **❓ Question:** Do we need complex adapters (like Resamplers) for SOTA performance?
*   **💡 Answer:** No. Molmo simplifies the architecture using a straightforward **MLP Projector** and focuses on high-quality data (PixMo).

**2. Detailed Data Flow**

$$
\text{Input Image} \xrightarrow{\text{ViT (e.g. SigLIP)}} \text{Features} \xrightarrow[\text{No Complex Adapter}]{\text{MLP Projector}} \text{Visual Tokens} \xrightarrow{\text{Concat}} \text{Unified LLM Context} \to \text{Output}
$$

> *Note: Unlike Flamingo, Molmo treats visual tokens as direct extensions of the text vocabulary in a unified decoder, relying on high-resolution patching and data quality rather than complex architecture.*

#### 4.5 SAM (Segment Anything Model)

**1. High-Level Concept**
*   **Flow:** $\text{Image} \to \text{Image Encoder} \to \text{Embedding} \leftarrow \text{Prompt Encoder (Points/Boxes/Text)} \to \text{Mask Decoder} \to \text{Segmentation Masks}$
*   **❓ Question:** How to create a promptable segmentation system with zero-shot generalization?
*   **💡 Answer:** Separate the heavy image encoding (run once) from the lightweight prompt encoding and mask decoding (run efficiently in real-time).

**2. Detailed Data Flow**

$$
\begin{aligned}
\text{Image} &\xrightarrow{\text{MAE ViT}} \text{Image Embedding} \\
\text{Prompts} &\xrightarrow{\text{Positional Encoding}} \text{Prompt Embedding} \\
\text{Embeddings} &\xrightarrow{\text{Transformer Decoder}} \text{Masks} + \text{IoU Scores}
\end{aligned}
$$

> *Note: The Image Encoder is heavy (ViT-H/L/B), while the Prompt Encoder and Mask Decoder are lightweight, allowing for real-time interaction in the browser after the image is processed once.*
